In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
from tqdm import tqdm

2026-03-01 07:27:42.711356: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-01 07:27:42.790111: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-01 07:27:43.256100: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-01 07:27:43.258504: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 07:27:44.849655: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

In [2]:
MODEL_PATH = "/home/yeezy/100xDevs/personality/Personality-Classifier/models/double_chin_model.h5"

DATA_PATH = "/home/yeezy/100xDevs/personality/Personality-Classifier/data/processed/jaw_dataset/valid"


IMG_SIZE = (224, 224)

In [3]:
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

2026-03-01 07:27:53.267116: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-01 07:27:53.268950: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 54, 54, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 26, 26, 128)       0

In [4]:
def prepare(image_path):
    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, IMG_SIZE)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    return img

In [ ]:
y_true = []
y_probs = []

for label in ["double_chin", "no_double_chin"]:

    folder = "/home/yeezy/100xDevs/personality/Personality-Classifier/data/processed/jaw_dataset/valid/" + label

    for file in tqdm(os.listdir(folder)):

        img_path = os.path.join(folder, file)
        img = prepare(img_path)

        if img is None:
            continue

        prob = model.predict(img, verbose=0)[0][0]

        y_probs.append(prob)

        if label == "double_chin":
            y_true.append(1)
        else:
            y_true.append(0)

print("Total samples:", len(y_true))

  0%|          | 0/1859 [00:00<?, ?it/s]

 15%|█▍        | 3386/22636 [05:21<34:56,  9.18it/s]  

In [ ]:
plt.hist(
    [np.array(y_probs)[np.array(y_true)==1],
     np.array(y_probs)[np.array(y_true)==0]],
    bins=50,
    label=["Double Chin", "No Double Chin"]
)

plt.legend()
plt.title("Probability Distribution")
plt.xlabel("Predicted Probability")
plt.ylabel("Count")
plt.show()
plt.savefig("/home/yeezy/100xDevs/personality/Personality-Classifier/research_exps/assets/cheekbone_probability_histogram.png")

🧠 2️⃣ How To Interpret ROC Curve
AUC Score Meaning
AUC	Meaning
0.5	Random
0.6–0.7	Weak
0.7–0.8	Acceptable
0.8–0.9	Strong
0.9+	Excellent

For personality system:
👉 You want at least 0.75+

In [ ]:
j_scores = tpr - fpr
best_index = np.argmax(j_scores)
best_threshold = thresholds[best_index]

print("Optimal Threshold:", best_threshold)

🟢 Cell 9 — Evaluate At Optimal Threshold

In [ ]:
y_pred = (np.array(y_probs) > best_threshold).astype(int)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))